# 01 — Bronze: clients

Same contract as `00_bronze_bids`: ingest as-is, validate the column set,
defer every judgement to Silver.

The client export carries two quirks worth noting but *not* fixing here — a
`2999-12-31` sentinel marking open-ended contracts, and literal `'null'`
strings where a value is absent. Both survive into Bronze untouched.

Same no-cluster-libraries approach as `00_bronze_bids`: pandas + openpyxl
reads the file, then it's converted to a Spark DataFrame.

In [0]:
CATALOG = "bronze"
SCHEMA = "bid"
VOLUME_PATH = "/Volumes/raw/bid/bids/clients.xlsx"
TABLE = f"{CATALOG}.{SCHEMA}.clients"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
# %uv pip install openpyxl

import pandas as pd
from pyspark.sql import functions as F

pdf_raw = pd.read_excel(VOLUME_PATH, sheet_name="Clients", dtype=str)
df_raw = spark.createDataFrame(pdf_raw)

df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(VOLUME_PATH))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(TABLE)
)

print(f"rows: {spark.table(TABLE).count()}")

In [0]:
EXPECTED = {
    "client_id", "contract_name", "status", "start_date", "end_date",
    "state", "city", "segment", "account_executive", "director",
    "manager", "coordinator",
}

actual = set(df_bronze.columns) - {"_ingested_at", "_source_file"}
missing, unexpected = EXPECTED - actual, actual - EXPECTED

if missing:
    raise ValueError(f"Columns missing from source: {sorted(missing)}")
if unexpected:
    print(f"WARNING — new columns absorbed, review Silver: {sorted(unexpected)}")

print("Schema check passed.")

In [0]:
%sql
select * from bronze.bid.clients